# KK1 – PGA Tour-spelarstatistik

Analys av spelarsäsonger på PGA Tour 2015–2022. Vi utforskar tre frågor: vad kännetecknar de bästa spelarna, hur spelet har förändrats över tid, och om det finns en trade-off mellan drive-längd och precision.

## 1. Inledning

PGA Tour är den dominerande professionella herr-golftouren i Nordamerika. Spelarna åker runt på en säsong av turneringar, samlar prispengar och poäng. För varje spelare och säsong registrerar touren detaljerad statistik: hur långt de slår från tee, hur ofta de träffar fairwayn, hur många greens de når i regulation, hur de puttar och vilket scoring-snitt de spelar på.

**Källa:** Kaggle – https://www.kaggle.com/datasets/robikscube/pga-tour-golf-data-20152022

**En rad i datasetet** representerar en spelare en säsong – t.ex. "Rory McIlroy 2018". Det innebär att samma spelare kan finnas på flera rader (en per säsong de spelat).

**Population:** Aktiva tour-spelare 2015–2022 som hade tillräckligt med rundor för att kvalificera för officiell statistik. Datasetet säger alltså inget om amatörgolfare, kvinnliga touren (LPGA), seniortouren eller spelare som missat sin tour-card.

I notebooken utforskar vi tre frågor som tråd:

1. **Vad kännetecknar de bästa spelarna?** Vilka stats skiljer dem som vinner från resten?
2. **Hur har spelet förändrats över tid?** Slår de längre nu än 2015? Är scoringen lägre?
3. **Finns det en trade-off mellan drive-längd och precision?** Är de längsta också raka, eller är längd och precision oförenliga?

Vi läser in datan, inspekterar och tvättar den, och svarar sedan på frågorna med visualiseringar.

## 2. Inläsning och inspektion

Först en mekanisk översikt av datan – form, kolumner, datatyper och saknade värden – innan vi tvättar något.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Läs in datasetet (en rad per spelare per turnering, 2015–2022)
df = pd.read_csv("data/ASA All PGA Raw Data - Tourn Level.csv")
df.shape

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
# Numerisk översikt av de kolumner som är relevanta för vår analys
df[["season", "n_rounds", "made_cut", "strokes",
    "sg_putt", "sg_arg", "sg_app", "sg_ott", "sg_t2g", "sg_total"]].describe()

In [ ]:
# Saknade värden per kolumn – vilka rader är ofullständiga och var?
df.isna().sum().sort_values(ascending=False).head(12)

**Observationer:**

- Datasetet har **36 864 rader** över **8 säsonger (2015–2022)** – varje rad är en spelares prestation i en specifik turnering.
- Spelaridentitet (`player`, `player_initial_last`), turnering (`tournament name`, `course`, `date`) och säsong (`season`) är kompletta.
- **Strokes Gained-kolumnerna** (`sg_putt`, `sg_app`, `sg_ott`, `sg_t2g`, `sg_total`) saknas i ~21 % av raderna (7 684 NaN). Dessa är hjärtat i vår analys, så vi behöver tänka igenom hur vi hanterar dem.
- Tre kolumner (`Unnamed: 2`, `Unnamed: 3`, `Unnamed: 4`) är helt tomma – CSV-artefakter som tas bort i tvätten.
- Kolumnerna `hole_DKP`, `streak_FDP` etc. är fantasy-sport-scoring (DraftKings, FanDuel). Inte relevanta för vår analys, ignoreras.
- `pos` saknas för ~42 % av raderna – troligen för spelare som missade cut (deras position registreras inte numeriskt). `made_cut` (0/1) är en renare indikator.

## 3. Datatvätt

Inspektionen visade tre saker att åtgärda – inget mer, inget mindre:

1. **Tre tomma `Unnamed`-kolumner** är CSV-artefakter och tas bort.
2. **Strokes Gained saknas i ~21 % av raderna.** Eftersom hela vår analys hänger på SG-statistiken filtrerar vi bort raderna där den saknas, snarare än att gissa värden. Vi noterar dock att vi tappar 7 684 rader och reflekterar i avslutningen över vad det innebär för slutsatserna.
3. **Fantasy-scoring-kolumnerna (DKP/FDP/SDP)** ignorerar vi men låter ligga – de gör ingen skada och att aktivt rensa dem skulle bara vara kosmetik.

Vi parsar inga textintervall (som i hundatan) eftersom alla numeriska kolumner redan är `int64` eller `float64`.

In [ ]:
# Steg 1: Släng de tre tomma Unnamed-kolumnerna
df = df.drop(columns=["Unnamed: 2", "Unnamed: 3", "Unnamed: 4"])
df.shape

In [ ]:
# Steg 2: Filtrera bort rader där Strokes Gained saknas
# Vi använder sg_total som referens – övriga sg_*-kolumner saknas på exakt samma rader
fore = len(df)
df = df[df["sg_total"].notna()].reset_index(drop=True)
efter = len(df)
print(f"Rader före: {fore}, efter: {efter}, borttagna: {fore - efter}")

In [ ]:
# Verifiera: inga NaN kvar i sg-kolumnerna, rätt antal rader per säsong
print("NaN i sg-kolumner efter tvätt:")
print(df[["sg_putt", "sg_arg", "sg_app", "sg_ott", "sg_t2g", "sg_total"]].isna().sum().to_string())
print()
print("Rader per säsong efter tvätt:")
print(df["season"].value_counts().sort_index().to_string())

**Resultat av tvätten:**

- Vi har kvar **29 181 rader** över alla 8 säsonger. Inga NaN kvar i sg-kolumnerna.
- Varje säsong är fortfarande välrepresenterad (mellan ~3 000 och ~4 700 rader per år), så jämförelser över tid är meningsfulla.
- Datan är nu redo att utforskas. Vi använder den både i sin tournament-form och – när frågan kräver det – aggregerar per spelare och säsong.

## 4. Visualiseringar

Här utforskar vi datan genom de tre frågorna vi formulerade i inledningen, plus en extra cell där vi prövar två diagramtyper för samma fråga.

### Fråga 1 – Vad kännetecknar de bästa spelarna?

För att svara behöver vi först definiera "bäst". Datan är på turneringsnivå, så vi aggregerar till **spelar-säsong**: hur många turneringar deltog spelaren i, hur många cuts klarade hen, och vad var medelvärdet för varje Strokes Gained-komponent över säsongen?

Sedan delar vi spelar-säsongerna i två grupper baserat på **cut-rate** (andel cuts klarade): topp-kvartilen (≥73 %) är "topp", resten är "övriga". Vi filtrerar bort säsonger med färre än 5 turneringar för att cut-rate ska vara meningsfull.

In [ ]:
# Aggregera till spelar-säsong-nivå
sg_kolumner = ["sg_putt", "sg_arg", "sg_app", "sg_ott", "sg_t2g", "sg_total"]
spelare_sasong = (
    df.groupby(["player", "season"])
      .agg(events=("made_cut", "size"),
           cuts=("made_cut", "sum"),
           **{kol: (kol, "mean") for kol in sg_kolumner})
      .reset_index()
)
spelare_sasong["cut_rate"] = spelare_sasong["cuts"] / spelare_sasong["events"]

# Filtrera till säsonger med >= 5 turneringar (stabil cut-rate)
spelare_sasong = spelare_sasong[spelare_sasong["events"] >= 5].reset_index(drop=True)
print(f"Spelar-säsonger efter filter: {len(spelare_sasong)}")
spelare_sasong.head()

In [ ]:
# Definiera topp-grupp = topp-kvartilen efter cut-rate
topp_grans = spelare_sasong["cut_rate"].quantile(0.75)
spelare_sasong["grupp"] = np.where(spelare_sasong["cut_rate"] >= topp_grans, "Topp", "Övriga")

print(f"Cut-rate-tröskel för topp-grupp: {topp_grans:.2%}")
print()
print("Gruppstorlekar:")
print(spelare_sasong["grupp"].value_counts().to_string())

In [ ]:
# Boxplot: jämför sg-komponenter mellan topp och övriga
komponenter = [
    ("sg_ott",   "Off the tee"),
    ("sg_app",   "Approach"),
    ("sg_arg",   "Around the green"),
    ("sg_putt",  "Putting"),
    ("sg_t2g",   "Tee to green"),
    ("sg_total", "Totalt"),
]

fig, axes = plt.subplots(2, 3, figsize=(12, 7), sharey=False)
for ax, (kol, namn) in zip(axes.flat, komponenter):
    data = [spelare_sasong.loc[spelare_sasong["grupp"] == g, kol]
            for g in ["Övriga", "Topp"]]
    ax.boxplot(data, labels=["Övriga", "Topp"])
    ax.axhline(0, color="gray", linewidth=0.8, linestyle="--")
    ax.set_title(namn)
    ax.set_ylabel("Strokes gained per runda")

fig.suptitle("Topp-spelare vinner strokes i hela spelet – mest från approach och tee-to-green",
             fontsize=13)
fig.tight_layout()
plt.show()

Skillnaden mellan grupperna syns tydligast i **approach** (sg_app) och **off the tee** (sg_ott) – topp-spelarnas medianer ligger klart över noll medan övriga ligger under. **Putting** (sg_putt) skiljer sig också, men spridningen är bredare i båda grupperna: bra puttning är en variabel även bland topp-spelare. **Around the green** (sg_arg) är den minsta skillnaden – kortspelet runt greenen verkar inte vara där eliten gör sin största vinst.

Tee-to-green-summan (sg_t2g) och totalen (sg_total) bekräftar bilden: i medeltal vinner topp-spelaren drygt en stroke per runda jämfört med övriga, och majoriteten av den vinsten kommer från det långa spelet (drive + approach), inte från greenen.

### Fråga 2 – Hur har spelet förändrats över tid?

Strokes Gained är mätt mot fältet vilket gör att medelvärdet alltid är ~0 per säsong – det är inte rätt mått för att se förändring över tid. Istället tittar vi på **strokes per runda** för spelare som klarat cut. Det är en mer direkt mätare av hur lågt det går att spela på tourens banor under en given säsong.

Vi visar medelvärdet per säsong med ett band för IQR (25:e–75:e percentilen) för att kombinera centraltendens med spridning.

In [ ]:
# Aggregera strokes per runda per säsong (endast made-cut-spelare)
df["strokes_per_runda"] = df["strokes"] / df["n_rounds"]
made_cut = df[df["made_cut"] == 1]

per_sasong = made_cut.groupby("season")["strokes_per_runda"].agg(
    medel="mean",
    q25=lambda s: s.quantile(0.25),
    q75=lambda s: s.quantile(0.75),
).round(2)
per_sasong

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(per_sasong.index, per_sasong["medel"], marker="o", color="steelblue",
        linewidth=2, label="Medel")
ax.fill_between(per_sasong.index, per_sasong["q25"], per_sasong["q75"],
                color="steelblue", alpha=0.18, label="IQR (Q1–Q3)")

# Annotera lägsta året
lagsta_ar = per_sasong["medel"].idxmin()
lagsta_v = per_sasong["medel"].min()
ax.annotate(f"Lägst: {lagsta_v:.2f} ({lagsta_ar})",
            xy=(lagsta_ar, lagsta_v),
            xytext=(lagsta_ar + 1.2, lagsta_v - 0.15),
            arrowprops=dict(arrowstyle="->", color="black", lw=1),
            fontsize=9)

ax.set_title("Strokes per runda har varierat ~1 slag mellan 2015 och 2022 – ingen tydlig nedåtgående trend",
             fontsize=12)
ax.set_xlabel("Säsong")
ax.set_ylabel("Strokes per runda (made-cut-spelare)")
ax.legend(loc="upper right")
fig.tight_layout()
plt.show()

Spelet har inte blivit konsekvent lättare under åren. Det fanns en svacka 2018–2019 där medelscoren låg ungefär ett halvt slag lägre per runda, men 2020 och framåt återgår scoren till nivåer som liknar 2015. IQR-bandet är ungefär lika brett varje år (~2 slag mellan Q1 och Q3) – spridningen mellan bra och sämre dagar har inte ändrats.

Ett viktigt förbehåll: PGA Tours kalender ändras år för år. Vilka banor som spelas påverkar scoring direkt – en svår banrotation ett år kan höja medelscoren oavsett spelarnas faktiska form. Vi kan inte separera "spelarna spelar bättre" från "banorna är lättare" med den här datan.

## 5. Avslutning